# Suite SKL — Agent guides

`AGENTS.md`, `llms.txt` and this package's skill, in `skill/`, teach AI agents what this package is for
and how to use it. Each language has its own skill, written in that language, on the same topics in the same order.
Stale guidance misleads more than missing guidance, so these cases check that the two skills have the same headings in
the same order, that this skill's complete programs run, and that every link in the guides resolves.

In [ ]:
import { spawnSync } from "node:child_process";
import { existsSync, mkdirSync, readdirSync, readFileSync, statSync, writeFileSync } from "node:fs";
import { join, relative, resolve } from "node:path";

import { assert } from "./support.js";

const ROOT = resolve("../..");
const SKILL = resolve("../skill");
const PYTHON = join(ROOT, "python3", "mbse", "Patterns", "skill");
const REPO = "https://github.com/pitaman71/mbse-patterns/blob/main/";

function walk(folder: string): string[] {
  return readdirSync(folder).flatMap((name) => {
    const path = join(folder, name);
    return statSync(path).isDirectory() ? walk(path) : [path];
  });
}

/** The files under `folder`, by relative path, in code-point order. */
function files(folder: string): [string, string][] {
  return walk(folder).map((path): [string, string] => [relative(folder, path).split("\\").join("/"), readFileSync(path, "latin1")])
    .sort(([a], [b]) => (a < b ? -1 : a > b ? 1 : 0));
}

/** Each Markdown file's headings, outside fenced blocks, in order, by relative path. */
function headings(folder: string): [string, string[]][] {
  return files(folder).filter(([name]) => name.endsWith(".md")).map(([name, text]): [string, string[]] =>
    [name, [...text.replace(/^```.*?^```$/gms, "").matchAll(/^#+ .*$/gm)].map((match) => match[0])]);
}

const GUIDES = [join(ROOT, "AGENTS.md"), join(ROOT, "llms.txt"),
  ...files(SKILL).filter(([name]) => name.endsWith(".md")).map(([name]) => join(SKILL, name))];

/** The complete programs in the skill: fenced blocks whose info string is exactly `language`. */
function blocks(language: string): string[] {
  return GUIDES.slice(2).flatMap((guide) => [...readFileSync(guide, "utf8").matchAll(new RegExp("^```" + language + "\\n(.*?)^```$", "gms"))]
    .map((match) => match[1] as string));
}

## SKL-01 · The two languages' skills have the same headings, in the same order

In [ ]:
assert(files(SKILL).some(([name]) => name === "SKILL.md") && files(PYTHON).some(([name]) => name === "SKILL.md"));
assert(JSON.stringify(headings(SKILL)) === JSON.stringify(headings(PYTHON)), JSON.stringify([headings(SKILL), headings(PYTHON)]));

## SKL-02 · Every complete program in the skill runs

In [ ]:
{
  const programs = blocks("typescript");
  assert(programs.length > 0, "the skill has no complete TypeScript program");
  mkdirSync(".build", { recursive: true });
  const failures: Record<number, string> = {};
  programs.forEach((code, i) => {
    const file = `.build/skill-${i}.ts`;
    writeFileSync(file, code);
    const check = spawnSync("npx", ["tsc", "--noEmit", "--strict", "--skipLibCheck", "--module", "nodenext",
      "--moduleResolution", "nodenext", "--target", "es2022", file], { encoding: "utf8", timeout: 300_000 });
    const run = check.status === 0 ? spawnSync("npx", ["tsx", file], { encoding: "utf8", timeout: 300_000 }) : check;
    if (run.status !== 0) failures[i] = `${run.stdout || ""}${run.stderr || ""}`.slice(-2000);
  });
  assert(Object.keys(failures).length === 0, JSON.stringify(failures));
}

## SKL-03 · Every link in the agent guides resolves, anchors included

In [ ]:
{
  /** GitHub's anchor for a heading. */
  const slug = (heading: string) => heading.trim().toLowerCase().replace(/[^\w\- ]/g, "").split(" ").join("-");
  /** A document's anchors: its headings' and its items' `<a id>`s. */
  const anchors = (text: string) => new Set([...[...text.matchAll(/^#+ (.*)$/gm)].map((m) => slug(m[1] as string)),
    ...[...text.matchAll(/<a id="([^"]+)"><\/a>/g)].map((m) => m[1] as string)]);
  const problems: string[] = [];
  let checked = 0;
  for (const guide of GUIDES) {
    for (const [, target] of readFileSync(guide, "utf8").matchAll(/\]\(([^)\s]+)\)/g) as IterableIterator<[string, string]>) {
      let path: string, base: string;
      const sibling = /^(?:mbse:([^/]+)|https:\/\/github\.com\/pitaman71\/([^/]+)\/blob\/main)\/(.*)$/.exec(target);
      if (sibling) [path, base] = [sibling[3] as string, join(ROOT, "..", (sibling[1] ?? sibling[2]) as string)]; // the sibling checkout, this one included
      else if (target.includes("://")) continue;
      else [path, base] = [target, resolve(guide, "..")];
      const [name, anchor] = path.split("#") as [string, string | undefined];
      const resolved = join(base, name);
      checked += 1;
      const label = `${relative(resolve(guide, ".."), guide)}: ${target}`;
      if (!existsSync(resolved)) problems.push(label);
      else if (anchor && !anchors(readFileSync(resolved, "utf8")).has(anchor)) {
        problems.push(label);
      }
    }
  }
  assert(problems.length === 0 && checked > 0, problems.join("; "));
}